# 01 · Data Exploration

Understand the HYG catalog before we render from it. We look at the
magnitude distribution (how many stars at each brightness), sky
coverage (any pole bias?), and compare a synthetic render to a real
Astrometry.net image when one is available.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
sns.set_theme(style='darkgrid')

from src.data.catalog import load_hyg_catalog
catalog = load_hyg_catalog(ROOT / 'data/catalogs/hygdata_v3.csv', mag_limit=8.0)
print(f'{len(catalog):,} stars with mag <= 8')
catalog.head()

## Magnitude distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(catalog['mag'], bins=60, color='#7c5cff', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Visual magnitude'); ax.set_ylabel('Star count')
ax.set_title('HYG catalog magnitude distribution (mag <= 8.0)')
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/01_mag_distribution.png', dpi=150)
plt.show()

## Sky coverage map (equal-area Mollweide)

In [ ]:
fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(111, projection='mollweide')
ra = np.deg2rad(catalog['ra_deg'].to_numpy() - 180.0)
dec = np.deg2rad(catalog['dec_deg'].to_numpy())
size = np.clip(8.0 - catalog['mag'], 0.2, 6.0)
ax.scatter(ra, dec, s=size, c='white', alpha=0.6, edgecolors='none')
ax.set_facecolor('#0b0d17')
ax.grid(True, color='#2a2f4f', alpha=0.4)
ax.set_title('Sky coverage of HYG stars (Mollweide)', color='black')
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/01_sky_coverage.png', dpi=150, facecolor='white')
plt.show()

## Synthetic vs real comparison
Render a synthetic field at known coordinates, then load a real
Astrometry.net image at *its* solved coordinates and put them side
by side. The renderer's job is to make these look statistically
interchangeable.

In [ ]:
from src.data.renderer import render_star_field
rng = np.random.default_rng(42)
synth = render_star_field(ra_center=85.0, dec_center=-2.0, field_width=30.0, rotation=0.0,
                          catalog=catalog, image_size=224, rng=rng)
fig, axes = plt.subplots(1, 2, figsize=(10, 5), facecolor='#0b0d17')
axes[0].imshow(synth); axes[0].set_title('Synthetic render @ Orion belt', color='white')
axes[1].text(0.5, 0.5, 'Drop a real image into\ndata/test_images/ to compare',
             ha='center', va='center', color='white', fontsize=11,
             transform=axes[1].transAxes)
axes[1].set_facecolor('#0b0d17')
for a in axes: a.set_xticks([]); a.set_yticks([])
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/01_synth_vs_real.png', dpi=150)
plt.show()